# 4. SHAP-based Explainability Analysis
## ICU XAI Research - Time-step Level Interpretability

This notebook provides explainability for ICU outcome predictions using SHAP:
- Compute SHAP values for each model
- Analyze feature importance per timestep
- Identify critical time windows for intervention
- Generate SHAP visualizations (force plots, summary plots)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from shap_utils import (
    compute_shap_values,
    aggregate_timestep_shap,
    extract_important_timesteps,
    feature_importance_per_timestep,
)


In [ ]:
# Load trained model and test data
import joblib

data_dir = Path.cwd().parent / 'data'
processed = pd.read_csv(data_dir / 'processed' / 'processed_data.csv')
feature_cols = [col for col in processed.columns if col not in ['RecordID', 'In-hospital_death', 'SAPS-I', 'SOFA']]

X = processed[feature_cols].values
y = processed['In-hospital_death'].astype(int).values

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

xgb_model = joblib.load(Path.cwd().parent / 'outputs' / 'models' / 'xgb_icuxai.pkl')
print('Loaded XGBoost model and test set shapes', X_test.shape, y_test.shape)


In [ ]:
# Compute SHAP values for XGBoost
background = X_train[:50]
shap_values_xgb = compute_shap_values(xgb_model, X_test, X_background=background)
print('SHAP values shape:', shap_values_xgb.shape)


In [ ]:
# Time-step importance from SHAP
important_timesteps = extract_important_timesteps(shap_values_xgb, top_k=5)
print(f"Most important timesteps: {important_timesteps}")


In [ ]:
# Feature importance per timestep
feature_prefixes = sorted({name.rsplit('_t', 1)[0] for name in feature_cols if '_t' in name})
timestep_ids = sorted({int(name.split('_t')[-1]) for name in feature_cols if '_t' in name})

n_features = len(feature_prefixes)
n_timesteps = len(timestep_ids)

if shap_values_xgb.ndim == 2 and shap_values_xgb.shape[1] == n_features * n_timesteps:
    shap_values_3d = shap_values_xgb.reshape(-1, n_timesteps, n_features)
else:
    raise ValueError('Cannot reshape SHAP values to (samples, timesteps, features)')

feature_importance_df = feature_importance_per_timestep(shap_values_3d, feature_prefixes)
print(feature_importance_df.head())


In [ ]:
# Feature importance per timestep
# TODO: Create dataframe of feature importance per timestep
# feature_importance_df = feature_importance_per_timestep(shap_values, feature_names)

In [ ]:
# SHAP summary visualization for XGBoost
try:
    shap.summary_plot(shap_values_xgb, features=X_test[:, :n_features], feature_names=feature_prefixes, show=False)
    plt.title('SHAP Summary for XGBoost Feature Importances')
    plt.show()
except Exception as e:
    print('Summary plot could not be rendered:', e)


In [ ]:
# Save SHAP analysis results
# TODO: Save SHAP values, plots, and interpretability reports to outputs/